In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [ ]:
df = pd.read_excel("../data/Training-data.xlsx")
print(df.head())

In [ ]:
test1 = pd.read_excel("../data/Test-data-1.xlsx")

In [177]:
test2 = pd.read_excel("../data/Test-data-2.xlsx")

## Data Preparation (non-learned steps)

### Irrelevant Features

In [ ]:
df1 = df.drop(columns=["tcprtt"])

### Missing Values

In [ ]:
# No empty values for now

### Duplicated Values

In [ ]:
df2 = df1.drop(columns=["is_ftp_login"])

### Feature Attribute Selection

In [ ]:
df3 = df2.drop(columns=['dwin'])

### Feature Engineering

In [ ]:
# Empty for now

### Class Imbalance

In [ ]:
# Empty for now

## Separating the predictors and the targets

In [ ]:
y = df3['label']
X = df3.drop(columns=['label'])

## Cross Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, KFold, ShuffleSplit

# 1. ShuffleSplit — Random sampling
#    Each split randomly selects test samples without preserving order.
#    Here, test_size=15 means 15 samples will be used for testing in each split.
ss = ShuffleSplit(
    n_splits=10,      # number of random splits
    test_size=15,     # absolute number of test samples per split
    random_state=4    # reproducibility
)

# 2. KFold — Ordered, non-stratified k-fold splitting
#    Splits the dataset into k contiguous folds (no shuffling by default).
kf = KFold(
    n_splits=10       # number of folds
    # shuffle=False by default
)

# 3. StratifiedKFold — Stratified k-fold splitting
#    Ensures that each fold has approximately the same class proportions as the full dataset.
skf = StratifiedKFold(
    n_splits=10       # number of folds
    # shuffle=False by default; set shuffle=True for randomised stratified folds
)

## k-NN Classifier

### Dependencies

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

### Step 1: Load the data

In [ ]:
y = df3['label']
X = df3.drop(columns=['label'])

### Step 2: Define transformers and build a pipeline 

In [ ]:
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=67
)

k_values = [1,3,7,11,17,21]

numeric_cols = X.select_dtypes(include=['number']).columns
categorical_cols = X.select_dtypes(include=['string']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), categorical_cols)
    ]
)

def make_knn_pipeline(k):
    return Pipeline([
        ("preprocessor", preprocessor),                 
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])

### Step 3: Evaluate different k-values with CV

In [178]:
cv_records = []
for k in k_values:
    print("Training on k = {}".format(k))
    pipe = make_knn_pipeline(k)
    scores = cross_val_score(pipe, X, y, cv=skf, scoring="accuracy", n_jobs=-1)
    cv_records.append({"k": k, "mean_cv_acc": scores.mean(), "std_cv_acc": scores.std()})

Training on k = 1
Training on k = 3
Training on k = 7
Training on k = 11
Training on k = 17
Training on k = 21


### Step 4: Select the champion model

In [180]:
cv_df = pd.DataFrame(cv_records).sort_values("mean_cv_acc", ascending=False).reset_index(drop=True)
print("\nCross-validation accuracy by k (training set):")
print(cv_df.to_string(index=False))

best_k = cv_df.loc[0, 'k']
print("Best k: {}".format(best_k))


Cross-validation accuracy by k (training set):
 k  mean_cv_acc  std_cv_acc
 7     0.994145    0.000310
11     0.994131    0.000326
17     0.993987    0.000350
21     0.993850    0.000460
 3     0.993514    0.000323
 1     0.991731    0.000541
Best k: 7


### Step 5: Retrain model with the full training dataset and evaluate on test datasets

In [ ]:
# pipe = make_knn_pipeline(best_k)
# pipe.fit(X, y)

# y_pred = pipe.predict(X_test)
# test_acc = accuracy_score(y_test, y_pred)

### Step 6: Display the classification scores and confusion matrix

In [ ]:
# print(f"Test accuracy (k={best_k}): {test_acc:.3f}")

# print("\nClassification report (test set):")
# print(classification_report(y_test, y_pred, digits=3))

### Step 7: Plot the confusion matrix for all k values for comparison

In [ ]:
# rows, cols = 2, 3
# fig, axes = plt.subplots(rows, cols, figsize=(15, 8))
# for idx, k in enumerate(k_values):
#     print("Training k = {}".format(k))
#     r, c = divmod(idx, cols)
#     pipe = make_knn_pipeline(k=k)
#     pipe.fit(X, y)
#     test1_pred = pipe.predict(test1)
#     test2_pred = pipe.predict(test2)
#     acc = accuracy_score(y_test, y_pred)

#     ConfusionMatrixDisplay.from_predictions(
#         y_test, y_pred,
#         cmap="Blues",
#         values_format="d",
#         ax=axes[r, c]
#     )
#     axes[r, c].set_title(f"k={k}  (Acc={acc:.2f})")

# plt.tight_layout()
# plt.show()

## Data Preparation (learned steps)
These steps must be done after data splitting.

### Scaling & Standardisation

In [ ]:
numeric_cols = df3.select_dtypes(include=['number']).columns
df3_standardised = df3.copy()
df3_standardised[numeric_cols] = (df3[numeric_cols] - df3[numeric_cols].mean()) / df3[numeric_cols].std()

### Data Types

In [ ]:
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

categorical_cols = df3_standardised.select_dtypes(include=['string']).columns
encoded = encoder.fit_transform(df3_standardised[categorical_cols])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(categorical_cols),
    index = df3_standardised.index
)

df3_encoded = df3_standardised.drop(columns=categorical_cols)
df3_encoded = pd.concat([df3_encoded, encoded_df], axis=1)